# Example: Using `HDFGClean` with `hdsims`

This notebook provides an example of how to use the `hdfgclean` package with maps generated by the `hdsims` package. It is broken up into two parts: 

**Part 1** is an example of how to run the full foreground (FG) cleaning procedure described in [MacInnis et. al. (2026)](https://arxiv.org/abs/XXXX.XXXXX) on a set of simulations (also described in that work) generated by the `hdsims` package. Here, we use simulations generated for a smaller $2^\circ \times 2^\circ$ patch of sky, as opposed to the default $10^\circ \times 10^\circ$ maps.  In particular, we will: 
- Download the simulations from the `sim_files_for_example_notebooks` repository on [github](https://github.com/CMB-HD/sim_files_for_example_notebooks)
- Save the configuration file used to initialize `HDFGClean`
- Run the FG cleaning procedure on the maps with `run_hdfgclean.py` (provided in the `hdfgclean` repository), which includes:
  - Applying matched filters to the maps to iteratively detect, measure, and remove point sources (CIB and radio galaxies) and tSZ clusters from the maps at each frequency
    - The catalogs of the sources and clusters that were removed from the maps, as well as the maps themselves after FG cleaning, will be saved (along with additional, intermediate output files); we will show you how to use the `HDFGClean` methods to load these files in "Part 2".
  - Matching the catalogs of detected sources and clusters to the true catalogs of all sources and clusters in the maps
  - Taking the power spectra of the FG-cleaned maps
- Compare the power spectra of the FG-cleaned maps to a set of pre-computed power spectra
- Plot the FG-cleaning results


**Part 2** provides a few additional, short examples of how to use the methods of `HDFGClean` after running the steps of part 1, including loading in (or calculating, if necessary): 
- The maps before and after FG cleaning and their power spectra
- Catalogs and maps of the detected point sources and clusters subtracted at each frequency
- The measured average CIB or radio spectral indices
- The catalogs produced by matching the detected point sources or clusters to the catalogs of all true point sources or clusters in the maps

Below, we import the python packages/modules needed to run this notebook:

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
from hdsims import fgcatalogs, plots
from hdfgclean import hdfgclean, hdfgclean_utils

---

# Part 1: FG cleaning

## Provide the paths to your directories

In the cell below, you **must** provide the paths to:
- your `hd_sims_dir` where the simulations will be saved (see [hdsims](https://github.com/CMB-HD/hdsims) for more information)
- an `output_dir` where the output from FG cleaning will be saved

**Note** that you will need about 6 GB of space to save the simulations, and about 3 GB to save the FG cleaning output.

In [ ]:
hd_sims_dir = 
output_dir = 

If you have moved (or made a copy of) this notebook into a different directory, provide the path to the `hdfgclean` repository (i.e., the directory that contains the readme file) below; otherwise, you can leave this cell unchanged:

In [ ]:
hdfgclean_repo_dir = None

## Download the simulations

Here, we will download the simulations used to run the FG cleaning in this example. If you are interested in generating new simulations yourself, refer to the examples and additional files provided in the [hdsims](https://github.com/CMB-HD/hdsims) github repository. 

The simulations we will run the FG cleaning on here are smaller (four-square-degree) versions of the default 100-square-degree HD simulations used in MacInnis et. al. (2026). They are 0.04 arcminute-resolution maps of the beam- and pixel-window-convolved sum of the lensed CMB, tSZ, kSZ, CIB, and radio sources, plus instrumental noise, at 90, 148, 219, and 277 GHz. We also need a second set of simulations, which are a different realization of the maps described above, to quantify the noise in the matched filter calculation (these are smaller versions of the ones described in the `hdfgclean` "[README](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#before-using-hdfgclean)" file).

In the following cell, we will print out intructions to download the necessary simulations from [github](https://github.com/CMB-HD/sim_files_for_example_notebooks).

In [ ]:
hdfgclean_utils.print_instructions_to_download_2x2hdsims(hd_sims_dir)

After following the instructions above, you must run the cell below to make sure the files were saved correctly; if they weren't, follow the instructions that are printed out:

In [ ]:
sim_files_saved = hdfgclean_utils.hdsims_for_example_are_saved(hd_sims_dir)
if not sim_files_saved:
    hdfgclean_utils.print_instructions_to_download_2x2hdsims(hd_sims_dir)
    print(f"\nThen, re-run this notebook cell.")

## Save configuration file for `HDFGClean`

In the following cell, we will save a `.yaml` configuration file with your `hd_sims_dir` and `output_dir`. We will also define and save the keyword arguments that must to passed to `HDFGClean` in order to use the simulations we downloaded above:

- The maps were generated by passing `width=2`, `height=2`, and `apod_width=0.2` to the `HDSims` class in `hdsims`, so we must also pass these arguments to `HDFGClean`. These are the width and height of the maps and the apodization width used to apodize them, respectively (all in degrees).
- We must pass the same keyword arguments in a dictionary of `noise_maps_kwargs`, since they were also used when generating the maps used to quantify the noise in the matched filter calculations. (We do not need to pass information about, e.g., the random seeds used to generate these maps since those were set to the defaults used in `hdfgclean`.)
- When the FG cleaning is run on larger maps, those maps will be divided into smaller patches, with the FG cleaning run on each individual patch (see the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#using-mpi-strongly-recommended) file for more information). This will not be done for our maps, but we still must pass `patch_apod_width=0.2` (same as the `apod_width`), since the default apodization width used on these patches is 0.25 degrees by default.


All other (optional) keyword arguments that can be passed to the `HDFGClean` class are set to the values used in MacInnis et. al. (2026) by default; see the `run_hdfgclean.ipynb` notebook provided in the `hdfgclean` github repository.

In [ ]:
# define the keyword arguments to pass to `HDFGClean`:
width = 2 # width (in degrees) of the sims being FG cleaned
height = width # height of sims
apod_width = 0.2 # apod. width (in degrees) to apodize sims
noise_maps_kwargs = {'width': width, 'height': height, 'apod_width': apod_width}
patch_apod_width = apod_width

# save a configuration file to run the FG cleaning using `reproduce_10x10.py`
config_file = os.path.join(output_dir, 'hdfgclean_example2x2.yaml')    
if not os.path.exists(config_file):
    hdfgclean.HDFGClean.save_config(config_file, hd_sims_dir, output_dir=output_dir,
                                    width=width, height=height, apod_width=apod_width,
                                    patch_apod_width=patch_apod_width, 
                                    noise_maps_kwargs=noise_maps_kwargs)

## Instructions to run the FG cleaning

Here we print out instructions to run the FG cleaning with the `run_hdfgclean` method of `HDFGClean`, initialized with your configuration file; this will be done by running the `run_hdfgclean.py` python script provided in the `hdfgclean` github repository.

**Note**: you do not need to use MPI to run the FG cleaning in this example, but if you are running on a cluster, we still recommend that you do **not** run the FG cleaning procedure (the second command printed out below) on the login nodes. For reference, all FG cleaning steps took about 6 hours in the `hbm-long-96core` [queue](https://rci.stonybrook.edu/HPC/faqs/seawulf-queues) on the Stony Brook SeaWulf cluster.

In [ ]:
hdfgclean_utils.print_hdfgclean_example_instructions(config_file, hdfgclean_repo_dir=hdfgclean_repo_dir)

## FG cleaning results

In the following cells, we initialize the `HDFGClean` class, and then make sure everything has been saved.

In [ ]:
hdfgcleanlib = hdfgclean.HDFGClean.from_config(config_file)

In [ ]:
fgclean_files_saved = hdfgclean_utils.all_2x2fgclean_files_are_saved(hdfgcleanlib, config_file, hdfgclean_repo_dir=hdfgclean_repo_dir)

Below, we compare the power spectra of your 90 and 148 GHz FG-cleaned maps to the following pre-computed power spectra:
- The power spectrum of residual CIB and radio sources at 90 and 148 GHz
- The residual tSZ power spectrum at 90 and 149 GHz
- The power spectrum of instrumental noise + kSZ + residual tSZ + residual CIB and radio sources ("FG + noise") at 90 and 148 GHz, and the coadded 90+148 GHz FG + noise power spectrum

In [ ]:
if fgclean_files_saved:
    hdfgclean_utils.compare_2x2_spectra(hdfgcleanlib, fdiff_tol=0.01)

## Plots

In the folowing cells, we will make versions of some of the plots shown in MacInnis et. al. (2026), using the FG cleaning results for this smaller $2^\circ \times 2^\circ$ patch of the sky. We will provide a brief summary of each plot; for further details, refer to the corresponding figure in that work.

### Measured point source fluxes

(corresponds to Figure 6 of MacInnis et. al. 2026) 

The _left panel_ compares the measured and true fluxes of 90 GHz CIB+radio point sources that were detected with SNR $\geq$ 4 and matched to a true source.

The _right panel_ shows the measured 148 GHz and 277 GHz fluxes of CIB sources used to calculate the average 277-to-148 CIB spectral index.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.flux_measurement_plot()

### Statistics of detected point sources

(corresponds to Figure 7 of MacInnis et. al. 2026) 

The _upper panels_ show, for 90 and 148 GHz (left and right panels, respectively), the number of bright sources detected with SNR $\geq$ 4 at the given frequency (yellow), dim sources detected at a different frequency (pink), and sources not detected (blue), all as a function of the true source flux; the number of false detections (as a function of measured flux) are shown in red.

The _lower panels_ show the power spectra of the CIB + radio maps at 90 and 148 GHz (left and right panels, respectively) before any FG cleaning (blue dotted), after subtracting bright sources detected with SNR $\geq$ 4 (yellow), and after subtracting all (bright and dim) sources (pink).

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.sources_plot()

### SZ cluster masses and redshifts

(corresponds to Figure 9 of MacInnis et. al. 2026) 

The true mass and redshift of all SZ clusters in the map that were detected with SNR $\geq$ 4 (red) and those that were not detected (blue).

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.cluster_mass_vs_redshift_plot()

### Maps before and after FG cleaning

(corresponds to Figure 10 of MacInnis et. al. 2026) 

The maps before (first column) and after (second column) FG cleaning. The first row shows 90 GHz CMB + FG (kSZ, tSZ, CIB, radio) maps. The second row shows the 90 GHz maps after subtracting out the frequency-independent CMB and kSZ for clarity, and the third row shows the corresponding maps at 148 GHz.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.plot_fgcleaned_maps_with_cmb()

### Maps of the tSZ, CIB, and radio sources

(corresponds to Figure 11 of MacInnis et. al. 2026) 

The tSZ + CIB + radio maps (i.e., with the CMB and kSZ subtracted out for clarity) at 90 and 148 GHz (top and bottom rows, respectively). The first and last columns show the maps before and after FG cleaning, respectively. The middle column shows maps of the measured CIB + radio point sources and tSZ clusters; these maps are subtacted from the maps in the first column to produce the maps in the last column.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.plot_fgcleaned_maps_without_cmb()

### Simulation-based residual FG power spectra at 90 and 148 GHz

(corresponds to Figure 2 of MacInnis et. al. 2026) 

Residual CIB+radio (blue), tSZ (green), and total FG + noise (dark red) power spectra of the 90 and 148 GHz (left and right panels, respectively) maps after FG cleaning; these are the same power spectra that you compared to the precomputed files earlier in this part. The previous estimate (from [MacInnis and Sehgal (2024)](https://arxiv.org/abs/2405.12220)) of the CMB-HD total FG + noise power spectra is shown in light red.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.plot_fgcleaned_spectra()

### Coadded 90 & 148 GHz simulation-based residual FG + noise power spectrum

(corresponds to Figure 1 of MacInnis et. al. 2026) 

The coadded 90 & 148 GHz power spectra of the residual FGs (kSZ, tSZ, CIB, radio) + instrumental noise. The simulation-based power spectrum is shown in dark red, and the previous estimate of MacInnis and Sehgal (2024) is shown in light red. The former is the same simulation-based power spectrum you compared to the precomputed files earlier in this part. 

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.plot_coadded_fgcleaned_spectra()

---

# Part 2: Additional examples

Note: you must complete "Part 1" above before proceeding.

Here, we show how the methods of the `HDFGClean` class can be used after the full FG cleaning procedure has been completed. We do not provide a complete description of each method used here; you should always refer to the documentation (i.e., the docstring written under the method definition in the relevant module of `hdfgclean`) of any method before using it (see the note below). For additional information, see the `hdfgclean` [readme](https://github.com/CMB-HD/hdfgclean/blob/main/README.md) file or refer to MacInnis et. al. (2026).

**Note**: we have ensured (when running `run_hdfgclean.py` in Part 1) that everything we use below has already been saved. In general, the methods of `HDFGClean` will perform whatever calculations are necessary in order to return whatever you have requested. 

For example, the `get_sim_power` method returns the power spectrum of a simulation; if the power spectrum has not been previously saved, then it will be calculated. This may require calculating the inverse mode-coupling matrix (used to correct for the effect of the map apodization), running the full FG-cleaning procedure (if this has not yet been done) if you request the power spectrum of a FG-cleaned map, or even generating the simulation itself if necessary. The `get_sim_power` method saves the power spectrum by default, but other methods will not save their calculations by default.

---

Below, we load in the 148 GHz maps before and after FG cleaning using the `get_sim` method of `HDFGClean`. 

The `HDFGClean.get_sim` accepts the same arguments as the corresponding method of `HDSims`; these arguments are passed directly to `HDSims.get_sim`. There are two new optional keyword arguments in `HDFGClean.get_sim`: `subtract_sources` and `subtract_clusters`.
- These are both `False` by default; in this case, the map returned by `HDSims.get_sim` is returned.
- If `subtract_sources=True`, the map of detected CIB and radio point sources for this frequency will be subtracted from the map returned by `HDSims.get_sim`.
  - This includes all bright point sources detected with SNR $\geq$ 4 in the map at this frequency, and sources that were not detected at this frequency, but were detected at 90 GHz (for radio sources) or 277 GHz (for CIB sources). We will refer to these point sources (i.e., both bright and dim) as the "detected sources" throughout this example.
- If `subtract_clusters=True`, the map of detected tSZ clusters will be subtracted from the map returned by `HDSims.get_sim`.




In [ ]:
freq = 148
map_before = hdfgcleanlib.get_sim(freq=freq, beam=True, noise=True)
map_after = hdfgcleanlib.get_sim(freq=freq, beam=True, noise=True, subtract_sources=True, subtract_clusters=True)

In [ ]:
plots.plot_maps([map_before, map_after], 
                labels=[f'{freq} GHz before FG cleaning', f'{freq} GHz after FG cleaning'], 
                grid=False, colorbar_ranges=[-300,300])

We can also load in and plot maps of all point sources and clusters that were subtracted from the 148 GHz map (or, in general, make them from the detected catalogs if they had not been saved):

In [ ]:
# remove the extra "padding" for apodization around the map edges
# by passing `include_apodized_region=False`:
subtracted_sources_map = hdfgcleanlib.get_map_of_subtracted_sources(freq, include_apodized_region=False)
subtracted_clusters_map = hdfgcleanlib.get_map_of_subtracted_clusters(freq, include_apodized_region=False)

plots.plot_maps([subtracted_sources_map, subtracted_clusters_map], 
                labels=[f'CIB+Radio sources\nremoved at {freq} GHz', f'tSZ clusters removed at {freq} GHz'], 
                grid=False, colorbar_ranges=[-50, 50])

Next, we load in and print out the 148 GHz detected point source and cluster catalogs. The catalogs are sorted by signal-to-noise ratio (SNR), with the highest-SNR detection in the first row.

Both of these catalogs have columns named `'RADeg`' and `'decDeg'` for the right ascension (R.A.) and declination (dec.), in degrees, of each detected source or cluster, and a column named `'SNR'` for the SNR of the detection.  For clusters, the R.A. and dec. coordinates are the location where the SNR of the detection was at its maximum, which we assume to be the center of the cluster.
- The detected cluster catalogs also have a column named `'y_c'` for the measured Compton-y parameter at the cluster center, and a column named `'template'` for the name of the cluster profile that produced the highest SNR detection for that cluster.
- The detected source catalogs also have a column named `'fluxmJy'` for the measured flux (in mJy) of the source at that frequency, and a column named `'component'` that, when possible, identifies the source as either `'cib'` or `'radio'`.

The catalogs also have other columns that are used for "book-keeping" while the FG cleaning is running; we do not print those out in this notebook for clarity.

In [ ]:
detected_sources = hdfgcleanlib.get_catalog_of_subtracted_sources(freq=freq, include_apodized_region=False)
detected_clusters = hdfgcleanlib.get_catalog_of_subtracted_clusters(include_apodized_region=False)

In [ ]:
print("detected clusters:")
detected_cluster_cols = ['RADeg', 'decDeg', 'y_c', 'SNR', 'template']
fgcatalogs.display_catalog(detected_clusters[detected_cluster_cols])

In [ ]:
print(f"{freq} GHz detected point sources:")
detected_source_cols = ['RADeg', 'decDeg', 'fluxmJy', 'SNR', 'component']
fgcatalogs.display_catalog(detected_sources[detected_source_cols])

You may have noticed that there are sources with negative SNR in the catalog. These are sources that were detected at 90 or 277 GHz, but not at 148 GHz; we assume that they are dim (at 148 GHz) radio or CIB sources, respectively, and remove them from the 148 GHz map using the measured average 90-to-148 radio or 277-to-148 CIB spectral index to calculate their 148 GHz flux. We calculate the SNR at the locations of these sources in the 148 GHz map *before* subtracting them; the SNR at some locations may be negative, e.g., if there is a tSZ cluster at the same location (which has a negative signal at 148 GHz). 

We can look at the measured average spectral indices, and the true indices calculated from all sources in the map:

In [ ]:
mean_cib_index, _ = hdfgcleanlib.get_spectral_index(freq, 'cib')
true_cib_index, _ = hdfgcleanlib.get_true_spectral_index(freq, 277, 'cib')
print(f"Measured average 277-to-148 CIB spectral index = {mean_cib_index:.2f}, true index = {true_cib_index:.2f}")

mean_radio_index, _ = hdfgcleanlib.get_spectral_index(freq, 'radio')
true_radio_index, _ = hdfgcleanlib.get_true_spectral_index(freq, 90, 'radio')
print(f"Measured average 90-to-148 radio spectral index = {mean_radio_index:.2f}, true index = {true_radio_index:.2f}")

We can count how many of the 148 GHz detected sources were identified as radio or CIB sources, and how many could not be identified because they were not matched to a 90 or 277 GHz detected source:

In [ ]:
total_num_sources = len(detected_sources)
num_cib = len(detected_sources[detected_sources['component'].eq('cib')])
num_radio = len(detected_sources[detected_sources['component'].eq('radio')])
num_unknown = len(detected_sources[detected_sources['component'].eq('unknown')])

print(f"{total_num_sources} sources were removed from the {freq} GHz map")
print(f"{num_cib} were identified as CIB, and {num_radio} were identified as radio sources; ")
print(f"{num_unknown} could not be identified, because they were not also found at 90 (if applicable) or 277 GHz")

The catalogs of all true CIB+radio point sources or tSZ clusters in the map can be loaded in using the `get_true_sources_catalog` or `get_true_clusters_catalog` methods, respectively. These catalogs have the same columns as described on the HD simulations page on [LAMBDA](https://lambda.gsfc.nasa.gov/simulation/ultrahigh_resolution_sims.html).

In [ ]:
all_true_sources = hdfgcleanlib.get_true_sources_catalog()
all_true_clusters = hdfgcleanlib.get_true_clusters_catalog()

In [ ]:
num_detected_sources = len(detected_sources)
num_detected_clusters = len(detected_clusters)
num_true_sources = len(all_true_sources)
num_true_clusters = len(all_true_clusters)

print(f"Removed {num_detected_sources} point sources out of {num_true_sources} true sources in the maps")
print(f"Removed {num_detected_clusters} clusters out of {num_true_clusters} true clusters in the maps")

Now, let's zoom in on the highest-SNR source in the maps before and after FG cleaning by looking at a $0.25^\circ \times 0.25^\circ$ region:

In [ ]:
# catalog row for the highest SNR source: 
highest_snr_source = detected_sources[detected_source_cols].iloc[0]
# get the region of maps to cut out:
cutout_ra_ctr = highest_snr_source['RADeg']
cutout_dec_ctr = highest_snr_source['decDeg']
cutout_width = 0.25 # degrees
cutout_height = cutout_width

In [ ]:
plots.plot_maps([map_before, map_after], 
                labels=[f'{freq} GHz before FG cleaning', f'{freq} GHz after FG cleaning'], 
                colorbar_ranges=[-300,300],
                ra_ctr=cutout_ra_ctr, dec_ctr=cutout_dec_ctr, plt_radius=cutout_width/2)

We can count how many detected and true point sources and clusters are within this region:

In [ ]:
# catalogs of detected and sources within the cut out region:
cutout_detected_sources = fgcatalogs.trim_catalog_positions(detected_sources, 
                                                            ra_ctr=cutout_ra_ctr, dec_ctr=cutout_dec_ctr, 
                                                            width=cutout_width, height=cutout_height)
cutout_detected_clusters = fgcatalogs.trim_catalog_positions(detected_clusters, 
                                                             ra_ctr=cutout_ra_ctr, dec_ctr=cutout_dec_ctr, 
                                                             width=cutout_width, height=cutout_height)
# catalogs of true and sources within the cut out region:
cutout_true_sources = fgcatalogs.trim_catalog_positions(all_true_sources, 
                                                        ra_ctr=cutout_ra_ctr, dec_ctr=cutout_dec_ctr, 
                                                        width=cutout_width, height=cutout_height)
cutout_true_clusters = fgcatalogs.trim_catalog_positions(all_true_clusters, 
                                                         ra_ctr=cutout_ra_ctr, dec_ctr=cutout_dec_ctr, 
                                                         width=cutout_width, height=cutout_height)


num_cutout_detected_sources = len(cutout_detected_sources)
num_cutout_detected_clusters = len(cutout_detected_clusters)
num_cutout_true_sources = len(cutout_true_sources)
num_cutout_true_clusters = len(cutout_true_clusters)

print("In the 0.25 deg. x 0.25 deg. region shown above:")
print(f"Removed {num_cutout_detected_sources} point sources out of {num_cutout_true_sources} true sources")
print(f"Removed {num_cutout_detected_clusters} clusters out of {num_cutout_true_clusters} true clusters")

In [ ]:
print("detected clusters in this region:")
fgcatalogs.display_catalog(cutout_detected_clusters[detected_cluster_cols])

In [ ]:
print("detected point sources in this region:")
fgcatalogs.display_catalog(cutout_detected_sources[detected_source_cols])

We can use the `match_to_true_sources_catalog` method to get the catalog of all detected 148 GHz point sources that were matched to a source in the catalog of all true 148 GHz point sources in the map; we will call this the "matched catalog". This method also returns a catalogs of all detected or true sources that weren't matched. The matched catalog has the same columns as the `detected_sources` catalog, and additional columns with the position and flux of each true source that was matched. There is also a `match_to_true_clusters_catalog` method for the tSZ clusters.

In [ ]:
matched_sources, unmatched_detected_sources, unmatched_true_sources = hdfgcleanlib.match_to_true_sources_catalog(freq)

# cut out the region we plotted above:
cutout_matched_sources = fgcatalogs.trim_catalog_positions(matched_sources, 
                                                           ra_ctr=cutout_ra_ctr, dec_ctr=cutout_dec_ctr, 
                                                           width=cutout_width, height=cutout_height)
# print out the columns in this order:
matched_sources_cols = ['SNR', 'RADeg', 'decDeg', 'true_RADeg', 'true_decDeg', 
                        'fluxmJy', 'true_fluxmJy', 'component', 'true_component']
fgcatalogs.display_catalog(cutout_matched_sources[matched_sources_cols])

We can also look at the map before removing clusters, but after removing point sources. For clarity, we will also remove the CMB and kSZ signals from each map before plotting it, and zoom in a little further.

In [ ]:
fg_components = ['tsz', 'cib', 'radio']
fgs_map_before = hdfgcleanlib.get_sim(freq=freq, beam=True, noise=True, components=fg_components)
fgs_map_middle = hdfgcleanlib.get_sim(freq=freq, beam=True, noise=True, components=fg_components, 
                                      subtract_sources=True, subtract_clusters=False)
fgs_map_after = hdfgcleanlib.get_sim(freq=freq, beam=True, noise=True, components=fg_components, 
                                     subtract_sources=True, subtract_clusters=True)
plots.plot_maps([fgs_map_before, fgs_map_middle, fgs_map_after], 
                labels=['Before FG cleaning', 'After sources, before clusters', 'After FG cleaning'], 
                colorbar_ranges=[-100,100], ncol=3,
                ra_ctr=cutout_ra_ctr, dec_ctr=cutout_dec_ctr, plt_radius=cutout_width/3)

We can see that initially, there was a bright point source at the same location as a tSZ cluster, which has been removed in the middle panel. However, the tSZ signal at that location leads to an under-estimation of the source flux; if you look carefully at that location in the last panel, you will notice a redder spot where we have under-subtracted that source.

Before taking the power spectrum of the FG-cleaned map, we mask locations where we have over- or under-subtracted a point source or cluster. Below, we will load in the 148 GHz mask, and plot it in the region shown above:

In [ ]:
mask = hdfgcleanlib.get_mask(freq)

In [ ]:
plots.plot_maps([fgs_map_after, mask], 
                labels=[f'{freq} GHz after FG cleaning', f'{freq} GHz mask'], 
                colorbar_ranges=[[-100,100], [0, 1]],
                ra_ctr=cutout_ra_ctr, dec_ctr=cutout_dec_ctr, plt_radius=cutout_width/2)

Finally, we can load in the power spectra of the maps (or, in general, calculate it, but we've already done this) using the `get_sim_power` method of `HDFGClean`. This is similar to the `get_sim` method, in the sense that it overrides the `HDSims.get_sim_power` method adding three new optional keyword arguments: `subtract_sources`, `subtract_clusters`, and `mask`, all of which are `False` by default. The first two have the same meaning as in the `HDFGClean.get_sim` method. If `mask=True`, the mask will be applied to the map before taking its power spectrum.

Below, we will plot power spectra of the realizations of the lensed CMB, kSZ, and instrumental noise in our 148 GHz map, along with the tSZ and CIB+radio simulation power spectra before and after FG cleaning. We also plot the total FG (kSZ, tSZ, CIB, radio) + noise before and after FG cleaning.

In [ ]:
# get the power spectra of the sim:
cmb_sim_power = hdfgcleanlib.get_sim_power(freq=freq, components=['cmb'], beam=True, 
                                           noise=False, bin_dl=True, bin_cl=False)
noise_sim_power = hdfgcleanlib.get_noise_sim_power(freq=freq, beam=True, bin_dl=True, bin_cl=False)
ksz_sim_power = hdfgcleanlib.get_sim_power(freq=freq, components=['ksz'], beam=True, 
                                           noise=False, bin_dl=True, bin_cl=False)
tsz_sim_power_before = hdfgcleanlib.get_sim_power(freq=freq, components=['tsz'], beam=True, 
                                                  noise=False, bin_dl=True, bin_cl=False)
tsz_sim_power_after = hdfgcleanlib.get_sim_power(freq=freq, components=['tsz'], beam=True, 
                                                 noise=False, bin_dl=True, bin_cl=False,
                                                 subtract_sources=True, subtract_clusters=True, mask=True)
cib_radio_sim_power_before = hdfgcleanlib.get_sim_power(freq=freq, components=['cib', 'radio'], beam=True, 
                                                        noise=False, bin_dl=True, bin_cl=False)
cib_radio_sim_power_after = hdfgcleanlib.get_sim_power(freq=freq, components=['cib', 'radio'], beam=True, 
                                                       noise=False, bin_dl=True, bin_cl=False,
                                                       subtract_sources=True, subtract_clusters=True, mask=True)
fg_noise_sim_power_before = hdfgcleanlib.get_sim_power(freq=freq, components=['ksz', 'tsz', 'cib', 'radio'], 
                                                       beam=True, noise=False, bin_dl=True, bin_cl=False)
fg_noise_sim_power_after = hdfgcleanlib.get_sim_power(freq=freq, components=['ksz', 'tsz', 'cib', 'radio'], 
                                                      beam=True, noise=False, bin_dl=True, bin_cl=False,
                                                      subtract_sources=True, subtract_clusters=True, mask=True)

# plot the power spectra:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.set_yscale('log')
ax.plot(cmb_sim_power['ells'], cmb_sim_power['dltt'], 
        label='Lensed CMB', color='k', lw=1.15)
ymin, ymax = ax.get_ylim() # set y-axis limits based on CMB spectrum
ax.plot(noise_sim_power['ells'], noise_sim_power['dltt'], 
        label='Instrumental noise', color='tab:gray', lw=1.15, ls='-.')
ax.plot(ksz_sim_power['ells'], ksz_sim_power['dltt'], 
        label='kSZ', color='tab:orange', lw=1.15, ls='-.')
ax.plot(tsz_sim_power_before['ells'], tsz_sim_power_before['dltt'], 
        label='tSZ before FG cleaning', color='tab:olive', lw=1.15, ls=':')
ax.plot(tsz_sim_power_after['ells'], tsz_sim_power_after['dltt'], 
        label='tSZ after FG cleaning', color='tab:green')
ax.plot(cib_radio_sim_power_before['ells'], cib_radio_sim_power_before['dltt'], 
        label='CIB+radio before FG cleaning', color='tab:cyan', lw=1.15, ls=':')
ax.plot(cib_radio_sim_power_after['ells'], cib_radio_sim_power_after['dltt'], 
        label='CIB+radio after FG cleaning', color='tab:blue')
ax.plot(fg_noise_sim_power_before['ells'], fg_noise_sim_power_before['dltt'], 
        label='Total FG + noise before FG cleaning', color='tab:pink', lw=1.25, ls=':')
ax.plot(fg_noise_sim_power_after['ells'], fg_noise_sim_power_after['dltt'], 
        label='Total FG + noise after FG cleaning', color='tab:red', lw=2)
ax.legend(fontsize=8, title=f'{freq} GHz', title_fontsize=9)
ax.set_ylim([ymin, ymax])
ax.set_xlim([0,20000])
plt.show()